# Comparing Prompt Versions and Creating a Decision Gate

We have two prompt versions:

- **v1** — original, no explicit structure guidance
- **v2** — adds structure but is overly conservative

We run both against the 20-scenario golden dataset and use a decision gate to determine whether v2 is safe to ship.

In [1]:
import os
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [2]:
load_dotenv()

True

In [3]:
provider = os.getenv("LLM_PROVIDER", "openai")
model_kwargs = {"model": "gpt-4o-mini", "temperature": 0.0}
if provider == "vocareum":
    model_kwargs |= {"base_url": "https://openai.vocareum.com/v1", "api_key": os.getenv("VOCAREUM_API_KEY")}
llm = ChatOpenAI(**model_kwargs)

In [4]:
with open("prompts/system_prompt_v1.md") as f:
    PROMPT_V1 = f.read()

In [5]:
with open("prompts/system_prompt_v2.md") as f:
    PROMPT_V2 = f.read()

In [6]:
print(PROMPT_V1)

You are RealThor, a Listing Supply Intelligence agent for a PropTech operating in Ciudaty.

Your task is to analyze demand and supply signals for a given region and property profile.

Use the available data to recommend whether the company should prioritize bringing more properties of that type to the platform.

Your recommendation must include:
- priority: high, medium, or low
- reasoning based on demand
- reasoning based on supply
- recommended action
- confidence: high, medium, or low

Do not make claims that are not supported by the provided data.



In [7]:
print(PROMPT_V2)

You are RealThor, a Listing Supply Intelligence agent for a PropTech operating in Ciudaty.

Your task is to analyze demand and supply signals for a given region and property profile, and produce a business-oriented recommendation for the Demand Intelligence team.

Use the available data to recommend whether the company should prioritize bringing more properties of that type to the platform. Focus on marketplace listing priority only — do not suggest iBuyer acquisition or property purchases.

Be conservative in your recommendations. When demand signals are not exceptionally strong or there is any meaningful supply already in the market, prefer monitor_opportunity over prioritize_new_listings. Only recommend prioritize_new_listings when demand is clearly outstanding and supply is critically low.

Your recommendation must include:
- priority: high, medium, or low
- demand_reasoning: explain the demand signal in business terms
- supply_reasoning: explain the supply situation and what it me

In [8]:
demand = pd.read_csv("data/demand_signals.csv")

In [9]:
supply = pd.read_csv("data/supply_snapshot.csv")

In [10]:
scenarios = pd.read_csv("data/eval_golden_dataset.csv")

## Output Schema and Structured LLM

In [11]:
class ListingRecommendation(BaseModel):
    """Structured recommendation from RealThor."""
    priority: Literal["high", "medium", "low"]
    demand_reasoning: str
    supply_reasoning: str
    recommended_action: Literal["prioritize_new_listings", "monitor_opportunity", "deprioritize"]
    confidence: Literal["high", "medium", "low"]

In [12]:
structured_llm = llm.with_structured_output(ListingRecommendation)

## Helper Functions

In [13]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]
    if d.empty or s.empty:
        return "No data available for the requested segment."
    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [14]:
def ask_realthor(region: str, property_type: str, bedrooms: int, prompt: str) -> ListingRecommendation:
    context = build_context(region, property_type, bedrooms)
    return structured_llm.invoke([
        SystemMessage(content=prompt),
        HumanMessage(content=context),
    ])

In [15]:
def evaluate_output(result: ListingRecommendation, row) -> dict:
    priority_match = result.priority == row["expected_priority"]
    action_match = result.recommended_action == row["expected_action"]
    passed = priority_match and action_match
    failure_reason = ""
    if not priority_match:
        failure_reason += f"priority: got '{result.priority}', expected '{row['expected_priority']}'; "
    if not action_match:
        failure_reason += f"action: got '{result.recommended_action}', expected '{row['expected_action']}'"
    return {
        "passed": passed,
        "priority": result.priority,
        "action": result.recommended_action,
        "failure_reason": failure_reason.strip("; "),
    }

## Run Evaluation — Both Versions

In [16]:
all_results = []

for version, prompt in [("v1", PROMPT_V1), ("v2", PROMPT_V2)]:
    for _, row in scenarios.iterrows():
        result = ask_realthor(row["region"], row["property_type"], row["bedrooms"], prompt)
        eval_result = evaluate_output(result, row)
        all_results.append({
            "scenario_id": row["scenario_id"],
            "prompt_version": version,
            **eval_result,
        })

In [17]:
results_df = pd.DataFrame(all_results)
display(results_df)

,scenario_id,prompt_version,passed,priority,action,failure_reason
0,G001,v1,True,high,prioritize_new_listings,
1,G002,v1,False,high,prioritize_new_listings,"priority: got 'high', expected 'low'; action: ..."
2,G003,v1,False,high,prioritize_new_listings,"priority: got 'high', expected 'medium'; actio..."
3,G004,v1,True,high,prioritize_new_listings,
4,G005,v1,False,high,prioritize_new_listings,"priority: got 'high', expected 'low'; action: ..."
5,G006,v1,True,high,prioritize_new_listings,
6,G007,v1,False,medium,monitor_opportunity,"priority: got 'medium', expected 'low'; action..."
7,G008,v1,True,high,prioritize_new_listings,
8,G009,v1,False,high,prioritize_new_listings,"priority: got 'high', expected 'low'; action: ..."
9,G010,v1,False,high,prioritize_new_listings,"priority: got 'high', expected 'medium'; actio..."


## Comparison Table

In [18]:
summary = results_df.groupby("prompt_version")["passed"].agg(
    total="count",
    passed="sum"
)
summary["pass_rate"] = summary["passed"] / summary["total"]
display(summary)

,total,passed,pass_rate
prompt_version,,,
v1,20,7,0.35
v2,20,9,0.45


## Regression Analysis

A regression is a scenario that passed in v1 but fails in the candidate version.

In [19]:
v1_passing = set(
    results_df[(results_df["prompt_version"] == "v1") & results_df["passed"]]["scenario_id"]
)

In [20]:
def find_regressions(version: str) -> set:
    version_failing = set(
        results_df[(results_df["prompt_version"] == version) & ~results_df["passed"]]["scenario_id"]
    )
    return v1_passing & version_failing

In [21]:
regressions_v2 = find_regressions("v2")

print(f"v2 regressions vs v1: {regressions_v2 or 'none'}")

v2 regressions vs v1: {'G008', 'G017', 'G006', 'G004'}


## Decision Gate

Accept a version if its pass rate is at least as good as v1 AND it introduces no regressions.

In [22]:
v1_rate = summary.loc["v1", "pass_rate"]
v2_rate = summary.loc["v2", "pass_rate"]

In [23]:
accept_v2 = v2_rate >= v1_rate and len(regressions_v2) == 0

print(f"v1 pass rate: {v1_rate:.0%}")
print(f"v2 pass rate: {v2_rate:.0%} | regressions: {len(regressions_v2)} | Decision: {'ACCEPT' if accept_v2 else 'REJECT'}")
print()

if accept_v2:
    print("Ship v2.")
else:
    print("Do not ship v2. Keep v1.")

v1 pass rate: 35%
v2 pass rate: 45% | regressions: 4 | Decision: REJECT

Do not ship v2. Keep v1.
